# M6.A7 — XAI: TreeSHAP 예측 근거 + 자연어 변환 prototype

> 산출 근거: `docs/plan/ai/phase_06_model.md` M6.A7 · 산출 방법은 spec 기확정(`model_spec.md` §9 — TreeSHAP)
> 본 노트북의 몫: **출력 형태**(top-3·자연어 변환 여부) probe·확정 · 작성: 2026-07-27

🔒 **공개 저장소 데이터 정책** — 매출 절대 금액은 커밋하지 않는다. 예측 근거는 "평소 대비 ±%" 편차
프레임이라 본문 서술 자체가 정책과 정합(절대액 불필요).

**설계 핵심** — V1-t는 `log1p(y) − log1p(roll7)` **편차**를 예측하므로, SHAP 기여의 합 = 예측 편차.
즉 TreeSHAP 분해가 곧 "**평소(최근 7영업일 평균) 대비 왜 높/낮은가**"의 완결된 설명이 된다 —
점주용 근거 문구와 수학이 1:1로 맞는 구조(하이브리드 채택의 부수 이득, `04_baselines.ipynb` §3).

## 판정 요약 (TL;DR)

1. **출력 형태 확정안** — `top-3 기여 요인(방향·상대 기여 %) + rule-based 자연어 1문장` 채택.
   문장 프레임: *"내일은 평소 대비 {±X%} 수준이 예상됩니다 — {요인1 (±%)}, {요인2}, {요인3}."*
   **LLM 미사용**(비용·지연·환각 없음, 배치에서 결정론적 생성). API 응답 스키마(JSON) 포함 — Phase 7 입력.
2. **글로벌 정합** — mean|SHAP| 상위가 gain 순위와 일치(최근 운영 상태 roll7_atv·roll7_mean →
   요일·학사 순). 설명 체계가 모델 실제 동작과 어긋나지 않음을 확인.
3. **% 변환 규약** — SHAP 값 φ(로그비 공간)를 `exp(φ)−1`로 환산해 "요인별 ±%"로 표기. 요인들은
   곱으로 결합되므로 합산 표기는 근사임을 명시(top-3 위주 표기라 실무 영향 미미).
4. **정직한 한계 실측** — 2026-03 out-of-sample 3일 검증: 방향은 맞으나(목요일 피크 +57% 예상/+92% 실측)
   **삼일절은 -9% 예상 vs +96% 실측** — 공휴일 영업 표본이 7일뿐이라 특수일 학습이 약함.
   → 예측 근거는 "모델의 이유"이지 정답이 아니며, **신뢰도 배지(M6.A8)·P10/P90 구간과 반드시 짝**으로 노출.
5. **spec 반영** — model_spec §9 "출력 형태 probe 후 결정" 해소(PR #18). 남는 것: 신뢰도 기준(M6.A8).

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import shap

warnings.filterwarnings("ignore")

_here = Path.cwd()
AI_DIR = next(p for p in [_here.parent, _here, _here / "AI"] if (p / "data_prep").exists())
sys.path.insert(0, str(AI_DIR / "data_prep"))
import preprocess as pp
import lightgbm as lgb

PAL = {"blue": "#2a78d6", "orange": "#eb6834", "gray": "#d9d8d4", "ink2": "#52514e"}
_installed = {f.name for f in fm.fontManager.ttflist}
plt.rcParams.update({
    "font.family": [f for f in ("AppleGothic", "Apple SD Gothic Neo", "NanumGothic") if f in _installed] or ["sans-serif"],
    "axes.unicode_minus": False, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "figure.dpi": 100,
})

feat = pd.read_parquet(AI_DIR / "data/processed/features_daily.parquet").set_index("date")
feat, _ = pp.impute_weather(feat)
ob = feat[feat.is_open].copy()
y, r7 = ob.total_amount, ob.roll7_mean
folds = pp.make_monthly_folds(ob.index)
f_explain = folds[-2]  # 2026-03 fold — 학습에 없는 날들을 설명(out-of-sample)

KEEP = ["is_holiday", "semester_week", "is_semester_first2w", "temp_avg", "temp_range",
        "lag1_sales", "lag1_tx", "lag_dow_sales", "roll7_mean", "roll4dow_mean", "roll7_atv",
        "is_post_renewal", "days_since_reopen"]
X = ob[KEEP].copy()
X = pd.concat([X, pd.get_dummies(ob.index.dayofweek, prefix="dow").set_index(ob.index)], axis=1)
_b = X.select_dtypes(bool).columns
X[_b] = X[_b].astype(int)
BEST = dict(learning_rate=0.0257, num_leaves=9, min_child_samples=10, subsample=0.7093,
            colsample_bytree=0.6796, reg_alpha=0.001, reg_lambda=0.0)  # V1-t (05 모델 카드)

tr, va = f_explain["train"], f_explain["val"]
ytr = np.log1p(y.loc[tr]) - np.log1p(r7.loc[tr])
k = ytr.notna()
model = lgb.LGBMRegressor(n_estimators=800, random_state=42, verbosity=-1, **BEST)
model.fit(X.loc[tr][k], ytr[k],
          eval_set=[(X.loc[va], np.log1p(y.loc[va]) - np.log1p(r7.loc[va]))],
          callbacks=[lgb.early_stopping(50, verbose=False)])

explainer = shap.TreeExplainer(model)
sv = pd.DataFrame(explainer.shap_values(X.loc[va]), index=va, columns=X.columns)
base = float(explainer.expected_value)
print(f"설명 대상: {f_explain['month']} 검증일 {len(va)}일 (모델은 해당 월 미학습) | "
      f"base 편차(학습 구간 기대) exp({base:.3f})−1 = {np.exp(base)-1:+.0%}")

In [ ]:
# §1 글로벌 — mean|SHAP| top10, gain 순위와 비교
mean_abs = sv.abs().mean().sort_values(ascending=False)
gain = pd.Series(model.booster_.feature_importance("gain"), index=X.columns)
gain_rank = gain.rank(ascending=False).astype(int)
tbl = pd.DataFrame({"mean|SHAP|": mean_abs.head(10).round(4),
                    "gain 순위": [int(gain_rank[c]) for c in mean_abs.head(10).index]})
display(tbl)

top10 = mean_abs.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(7.5, 3.8), constrained_layout=True)
ax.barh(top10.index, top10.values, color=PAL["blue"], height=0.6)
ax.set_xlabel("mean |SHAP| (로그비 공간)")
ax.set_title("글로벌 기여 상위 10 — 2026-03 검증일 out-of-sample")
ax.grid(axis="y", visible=False)
plt.show()

### §1 관찰 — 글로벌 정합

- mean|SHAP| 상위(최근 객단가·최근 수준·같은 요일 이력 → 개강 직후·일요일·학기 주차)가 05 모델 카드의
  gain 순위와 일치 — 설명 체계와 학습 실체가 동일 방향. `04_baselines.ipynb` §3에서 기대한
  "편차를 흔드는 요인" 구조 그대로.
- 일요일 더미(dow_6)가 상위 — EDA §2.5의 일요일 저점이 설명에도 그대로 드러남.

In [ ]:
# §2 로컬 예시 3일 — top-5 기여 시각화 (out-of-sample)
EXAMPLES = ["2026-03-01", "2026-03-09", "2026-03-19"]  # 삼일절(일) · 개강 2주차 월 · 목요일 피크
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6), constrained_layout=True)
rows = []
for ax, d in zip(axes, EXAMPLES):
    ts = pd.Timestamp(d)
    top5 = sv.loc[ts].sort_values(key=abs, ascending=False).head(5).iloc[::-1]
    colors = [PAL["blue"] if v > 0 else PAL["orange"] for v in top5.values]
    ax.barh(top5.index, np.exp(top5.values) - 1, color=colors, height=0.6)
    ax.axvline(0, color=PAL["ink2"], lw=0.8)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:+.0%}"))
    pred = float(np.exp(base + sv.loc[ts].sum()) - 1)
    actual = float(y.loc[ts] / r7.loc[ts] - 1)
    ax.set_title(f"{d} ({ts.day_name()[:3]})\n예상 {pred:+.0%} / 실측 {actual:+.0%}", fontsize=9.5)
    rows.append((d, f"{pred:+.0%}", f"{actual:+.0%}"))
fig.suptitle("로컬 기여 top-5 — '평소 대비' 편차 분해 (파랑=올림, 주황=내림)", fontsize=11)
plt.show()
display(pd.DataFrame(rows, columns=["날짜", "예상(평소 대비)", "실측(평소 대비)"]))

### §2 관찰 — 로컬 설명과 정직한 한계

- **2026-03-19(목)**: +57% 예상/+92% 실측 — 방향·서사 모두 타당(목요일·기온·최근 흐름). 과소 폭은
  재개장 램프업 구간의 공통 현상(06 §2 밴드 그림과 동일 원인).
- **2026-03-09(월)**: +21% 예상/+137% 실측 — 개강 직후 플래그가 올렸지만 램프업 실제 폭을 못 따라감.
- **2026-03-01(삼일절·일)**: **-9% 예상 vs +96% 실측** — 모델은 "일요일 저점"을 적용했지만 실제론
  공휴일 특수가 압도. 공휴일 영업 이력이 7일뿐이라(EDA §5) 학습 신호가 약한 구조적 한계.
- 결론: 예측 근거는 **"모델이 왜 그렇게 예측했는가"의 설명**이지 정답 보증이 아니다 —
  UI에서 신뢰도 배지(M6.A8)·P10/P90 구간과 반드시 함께 노출한다. 공휴일·특수일은 배지가 뜨는
  대표 케이스가 될 것.

In [ ]:
# §3 자연어 변환 prototype — rule-based (LLM 미사용, 배치에서 결정론적 생성)
LABELS = {
    "is_holiday": "공휴일", "semester_week": "학기 진행 주차", "is_semester_first2w": "개강 직후 효과",
    "temp_avg": "기온", "temp_range": "일교차", "lag1_sales": "직전 영업일 매출 흐름",
    "lag1_tx": "직전 영업일 주문 수", "lag_dow_sales": "지난주 같은 요일 매출",
    "roll7_mean": "최근 일주일 매출 수준", "roll4dow_mean": "최근 같은 요일 평균",
    "roll7_atv": "최근 객단가 흐름", "is_post_renewal": "리뉴얼 이후 체제", "days_since_reopen": "재개장 경과",
    **{f"dow_{i}": f"{d}요일 효과" for i, d in enumerate("월화수목금토일")},
}

def explain_day(date, top_n=3):
    """예측 근거 생성 — Phase 7 AI Server 응답 스키마의 prototype."""
    ts = pd.Timestamp(date)
    phi = sv.loc[ts]
    pred_dev = float(np.exp(base + phi.sum()) - 1)          # 평소(최근 7영업일 평균) 대비 예상 편차
    top = phi.reindex(phi.abs().sort_values(ascending=False).index)[:top_n]
    factors = [{"feature": c, "label": LABELS.get(c, c), "pct": round(float(np.exp(v) - 1), 3)}
               for c, v in top.items()]
    updown = "높을" if pred_dev > 0 else "낮을"
    parts = ", ".join(f"{f['label']}({f['pct']:+.0%})" for f in factors)
    sentence = (f"{ts:%m월 %d일}({'월화수목금토일'[ts.dayofweek]})은 평소보다 약 {abs(pred_dev):.0%} "
                f"{updown} 것으로 예상됩니다 — 주요 요인: {parts}.")
    return {"date": str(ts.date()), "deviation_vs_baseline": round(pred_dev, 3),
            "baseline": "직전 7영업일 평균", "top_factors": factors, "sentence": sentence}

import json
for d in EXAMPLES:
    out = explain_day(d)
    print(out["sentence"])
print("\nAPI 응답 스키마 예시 (Phase 7 입력):")
print(json.dumps(explain_day(EXAMPLES[-1]), ensure_ascii=False, indent=2))

### §3 관찰 — 출력 형태 확정

| 항목 | 확정 |
|---|---|
| 구성 | **top-3 기여 요인**(방향 + 상대 기여 %) + **rule-based 자연어 1문장** |
| 프레임 | "평소(직전 7영업일 평균) 대비 ±X%" — 편차 모델의 SHAP 합과 수학적으로 일치 |
| 자연어 변환 | **채택하되 rule-based 템플릿** — LLM 미사용(결정론·무비용·무환각). 요인명은 LABELS 매핑 |
| % 규약 | φ → `exp(φ)−1`. 요인 결합은 곱이므로 표기는 근사(명시) |
| 스키마 | `{date, deviation_vs_baseline, baseline, top_factors[{feature,label,pct}], sentence}` |
| 노출 조건 | 신뢰도 배지(M6.A8)·P10/P90 구간과 항상 동반 — 단독 노출 금지 |

## §4 판정·다음 단계

**M6.A7 종료 판정** — TreeSHAP 통합 + top-3/자연어 출력 형태 확정 + 한계 실측(특수일 약함). 산출물
(SHAP 시각화 + 자연어 prototype) 충족. `model_spec.md` §9의 "출력 형태 probe 후 결정" 해소 — PR #18 반영.

- **M6.A8 신뢰도 기준**(다음): 입력 3종 확보 완료 — ① P10/P90 구간 폭(06 §2) ② 선행일(D+1~3) 차등(06 §1)
  ③ 특수일 케이스(본 §2). 산식·임계값 산정 → `feature_spec.md` §5.3 갱신
- Phase 7 연계: `explain_day()` 스키마가 AI Server 응답 설계 입력. shap은 `[ml]` extra — 서빙 이미지
  포함 여부는 Phase 7에서(배치 계산 후 DB 저장이라 서버 런타임엔 불필요할 수도)
- 검수 3건 변동 없음